In [1]:
import matplotlib as plt
import torch
import numpy as np
import tqdm
import pandas as pd
import seaborn as sns
import time


In [3]:
def plot_cv_indices(cv, X, y, group, ax, n_splits, lw=10):
    if X is None:
        X = np.zeros((len(y), 1))  # Dummy data array, just for plotting purposes
    
    if ax is None:
        fig, ax = plt.subplots(); fig.set_figheight(5);fig.set_figwidth(20)
    
    cmap_data = plt.cm.Paired
    cmap_cv = plt.cm.coolwarm
    """Create a sample plot for indices of a cross-validation object."""
    use_groups = "Group" in type(cv).__name__
    groups = group if use_groups else None
    # Generate the training/testing visualizations for each CV split
    for ii, (tr, tt) in enumerate(cv.split(X=X, y=y, groups=groups)):
        # Fill in indices with the training/test groups
        indices = np.array([np.nan] * len(X))
        indices[tt] = 1
        indices[tr] = 0
        
        # Visualize the results
        ax.scatter(range(len(indices)),[ii + 0.5] * len(indices),c=indices,marker="_",lw=lw,cmap=cmap_cv,vmin=-0.2,vmax=1.2,)

    # Plot the data classes and groups at the end
    ax.scatter(range(len(X)), [ii + 1.5] * len(X), c=y, marker="_", lw=lw, cmap=cmap_data)
    #ax.scatter(range(len(X)), [ii + 2.5] * len(X), c=group, marker="_", lw=lw, cmap=cmap_data)

    # Formatting
    yticklabels = list(range(n_splits)) + ["class"]
    ax.set(yticks=np.arange(n_splits + 1) + 0.5,yticklabels=yticklabels,xlabel="Sample index",ylabel="CV iteration",ylim=[n_splits + 2.2, -0.2])
    ax.set_title("{}".format(type(cv).__name__), fontsize=15)
    
    ax.autoscale(enable=True, axis='x', tight=True);ax.autoscale(enable=True, axis='y', tight=False)
    return ax

In [2]:
classes = {
    0: 47, 1: 23, 2: 27, 3: 34, 4: 43, 5: 5, 6: 41, 7: 14, 8: 46, 9: 22, 10: 37, 11: 38, 12: 24, 13: 3, 14: 12, 15: 13,
    16: 9, 17: 20, 18: 0, 19: 33, 20: 30, 21: 29, 22: 44, 23: 48, 24: 25, 25: 4, 26: 39, 27: 49, 28: 40, 29: 6, 30: 7,
    31: 32, 32: 26, 33: 31, 34: 2, 35: 17, 36: 10, 37: 1, 38: 11, 39: 8, 40: 42, 41: 21, 42: 28, 43: 19, 44: 18, 45: 45,
    46: 36, 47: 35, 48: 15, 49: 16

}
classes = list(classes.values())
#print(classes)
sound_dictionary = {
    0: "dog", 1: "rooster", 2: "pig", 3: "cow", 4: "frog", 5: "cat", 6: "hen", 7: "insects", 8: "sheep", 9: "crow",
    10: "rain", 11: "sea_waves", 12: "crackling_fire", 13: "crickets", 14: "chirping_birds", 15: "water_drops", 16: "wind",
    17: "pouring_water", 18: "toilet_flush", 19: "thunderstorm", 20: "crying_baby", 21: "sneezing", 22: "clapping", 23: "breathing",
    24: "coughing", 25: "footsteps", 26: "laughing", 27: "brushing_teeth", 28: "snoring", 29: "drinking_sipping", 30: "door_wood_knock",
    31: "mouse_click", 32: "keyboard_typing", 33: "door_wood_creaks", 34: "can_opening", 35: "washing_machine", 36: "vacuum_cleaner",
    37: "clock_alarm", 38: "clock_tick", 39: "glass_breaking", 40: "helicopter", 41: "chainsaw", 42: "siren", 43: "car_horn",
    44: "engine", 45: "train", 46: "church_bells", 47: "airplane", 48: "fireworks", 49: "hand_saw"
}
decoded_labels = []
for key in classes:
    decoded_labels.append(sound_dictionary.get(key, "Key not found"))
print(decoded_labels)
print(len(decoded_labels))
decoded_labels_esc10 = []
encoded_labels_esc10 = {0:"chainsaw",
                        1:"clock_tick",
                        2:"crackling_fire",
                        3:"crying_baby",
                        4:"dog",
                        5:"helicopter",
                        6:"rain",
                        7:"rooster",
                        8:"sea_waves",
                        9:"sneezing",}
for key in encoded_labels_esc10:
    decoded_labels_esc10.append(encoded_labels_esc10.get(key, "Key not found"))


def create_confusionmatrix(model,loader,num_class,device):
    nb_classes = num_class
    confusion_matrix = np.zeros((nb_classes, nb_classes))
    #print(f"{confusion_matrix.shape}\t{nb_classes}")
    model.eval()  # Set the model to evaluation mode

    with torch.no_grad():
        for i, (inputs, classes) in enumerate(tqdm(loader)):
            #print(i)
            inputs = inputs.to(device)
            classes = classes.to(device)
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            for t, p in zip(classes.view(-1), preds.view(-1)):
                    confusion_matrix[t.long(), p.long()] += 1
    
    fig = plt.figure(figsize=(15,10),dpi=100)
    class_names = decoded_labels
    if num_class == 10:
        class_names.clear()
        class_names = decoded_labels_esc10
        
    df_cm = pd.DataFrame(confusion_matrix, index=class_names, columns=class_names).astype(int)
    # Create a custom annotation array where 0s are replaced with empty strings
    annot = df_cm.map(lambda x: '' if x == 0 else '{:.0f}'.format(x))  #hide zeros
    
    heatmap = sns.heatmap(df_cm, annot=annot, fmt="s",annot_kws={"size": 10},xticklabels=True, yticklabels=True,cmap="coolwarm")#,linewidths=0.2, linecolor='black')
    
    heatmap.yaxis.set_ticklabels(heatmap.yaxis.get_ticklabels(), rotation=0, ha='right',fontsize=10)
    heatmap.xaxis.set_ticklabels(heatmap.xaxis.get_ticklabels(), rotation=90, ha='center',fontsize=10)
    plt.ylabel('True label'); plt.xlabel('Predicted label');
    plt.title(f"Confusion Matrix with {np.sum(confusion_matrix)} test samples")
    return fig
#pic = create_confusionmatrix(model,test_dataloader)
#create_confusionmatrix(model,train_dataloader,num_class=10);

['airplane', 'breathing', 'brushing_teeth', 'can_opening', 'car_horn', 'cat', 'chainsaw', 'chirping_birds', 'church_bells', 'clapping', 'clock_alarm', 'clock_tick', 'coughing', 'cow', 'crackling_fire', 'crickets', 'crow', 'crying_baby', 'dog', 'door_wood_creaks', 'door_wood_knock', 'drinking_sipping', 'engine', 'fireworks', 'footsteps', 'frog', 'glass_breaking', 'hand_saw', 'helicopter', 'hen', 'insects', 'keyboard_typing', 'laughing', 'mouse_click', 'pig', 'pouring_water', 'rain', 'rooster', 'sea_waves', 'sheep', 'siren', 'sneezing', 'snoring', 'thunderstorm', 'toilet_flush', 'train', 'vacuum_cleaner', 'washing_machine', 'water_drops', 'wind']
50


In [3]:
import numpy as np
import seaborn as sns




def create_confusionmatrix2(model,loader,num_class,device):
    nb_classes = num_class
    confusion_matrix = np.zeros((nb_classes, nb_classes))
    #print(f"{confusion_matrix.shape}\t{nb_classes}")
    model.eval()  # Set the model to evaluation mode

    with torch.no_grad():
        for i, (inputs, classes) in enumerate(tqdm(loader)):
            #print(i)
            inputs = inputs.to(device)
            classes = classes.to(device)
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            for t, p in zip(classes.view(-1), preds.view(-1)):
                    confusion_matrix[t.long(), p.long()] += 1
    
    fig = plt.figure(figsize=(15,10),dpi=100)
    class_names = decoded_labels
    if num_class == 10:
        class_names.clear()
        class_names = decoded_labels_esc10
       
    df_cm = pd.DataFrame(confusion_matrix, index=class_names, columns=class_names).astype(int)
    vmax = pd.DataFrame.sum(df_cm, axis=1)
    vmax = vmax.max(axis=0)
    #print(vmax)
    
    
    # Create a custom annotation array where 0s are replaced with empty strings
    annot = df_cm.map(lambda x: '' if x == 0 else '{:.0f}'.format(x))  #hide zeros
    off_diag_mask = np.eye(*confusion_matrix.shape, dtype=bool)

    heatmap = sns.heatmap(df_cm, annot=annot,mask=~off_diag_mask, fmt="s",annot_kws={"size": 10},xticklabels=True, yticklabels=True,cmap="Blues",vmin=0,vmax=vmax)#,linewidths=0.2, linecolor='black')

    #sns.heatmap(cf_matrix, annot=True, mask=off_diag_mask, cmap='OrRd', vmin=vmin, vmax=vmax, cbar_kws=dict(ticks=[]))

    heatmap = sns.heatmap(df_cm, annot=annot, mask=off_diag_mask, fmt="s",annot_kws={"size": 10},xticklabels=True, yticklabels=True,cmap="OrRd",vmin=0,vmax=vmax)#,linewidths=0.2, linecolor='black')
    
    heatmap.yaxis.set_ticklabels(heatmap.yaxis.get_ticklabels(), rotation=0, ha='right',fontsize=10)
    heatmap.xaxis.set_ticklabels(heatmap.xaxis.get_ticklabels(), rotation=90, ha='center',fontsize=10)
    plt.ylabel('True label'); plt.xlabel('Predicted label');
    plt.title(f"Confusion Matrix with {np.sum(confusion_matrix)} test samples")
    return fig

In [6]:

_start_time = time.time()

def tic():
    global _start_time 
    _start_time = time.time()

def tac():
    t_sec = round(time.time() - _start_time)
    (t_min, t_sec) = divmod(t_sec,60)
    (t_hour,t_min) = divmod(t_min,60) 
    print('Time passed: {}hour:{}min:{}sec\t'.format(t_hour,t_min,t_sec))

In [1]:
def log_state(writer,epoch,train_loss,train_acc,test_loss,test_acc,train_correct,correct,cfg):
    #Note: Tensorboard Logging
        #writer.add_scalar("Loss/Train", train_loss, epoch)
        #writer.add_scalar("Loss/Test", test_loss, epoch)
        #writer.add_scalar("Accuracy/Train", train_acc*100, epoch)
        #writer.add_scalar(tag = "Accuracy/Test", scalar_value = test_acc*100,global_step =  epoch)
        
        writer.add_scalars("Loss",{"Train/"+str(cfg.fold):train_loss},epoch)
        writer.add_scalars("Loss",{"Test/"+str(cfg.fold):test_loss},epoch)
    
                
        writer.add_scalars("Accuracy",{"Train/"+str(cfg.fold):train_acc*100},epoch)
        writer.add_scalars("Accuracy",{"Test/"+str(cfg.fold):test_acc*100},epoch)

        #writer.add_scalar('Loss/test', test_loss, epoch)
        #writer.add_scalar('Loss/train', train_loss, epoch)
        #writer.add_scalar('Accuracy/test', test_acc*100, epoch)
        #writer.add_scalar('Accuracy/train', train_acc*100, epoch)
        #writer.add_scalar('AccNum/train', train_correct, epoch) 
        #writer.add_scalar('AccNum/test', correct, epoch) 

        writer.add_scalars("AccuracySamples",{"Train/"+str(cfg.fold):train_correct},epoch)
        writer.add_scalars("AccuracySamples",{"Test/"+str(cfg.fold):correct},epoch)
        # writer.add_scalars("Train Loss/Accuracy",{'Loss':train_loss},epoch)
        # writer.add_scalars("Train Loss/Accuracy",{'Accuracy':train_acc*100},epoch)
        # 
        # writer.add_scalars("Test Loss/Accuracy",{'Loss':test_loss},epoch)
        # writer.add_scalars("Test Loss/Accuracy",{'Accuracy':test_acc*100},epoch)
            
        
        writer.add_scalars("Loss/"+str(cfg.fold),{"Train":train_loss},epoch)
        writer.add_scalars("Loss/"+str(cfg.fold),{"Test":test_loss},epoch)
        writer.add_scalars("Accuracy/"+str(cfg.fold),{"Train":train_acc*100},epoch)
        writer.add_scalars("Accuracy/"+str(cfg.fold),{"Test":test_acc*100},epoch)
        writer.add_scalars("AccuracySamples/"+str(cfg.fold),{"Train":train_correct},epoch)
        writer.add_scalars("AccuracySamples/"+str(cfg.fold),{"Test":correct},epoch)

    
    
        #archive
            
        # writer.add_scalars("Loss",{"Train/"+str(cfg.fold):train_loss},epoch,description="loss at every epoch")
        # writer.add_scalars("Loss",{"Test/"+str(cfg.fold):test_loss},epoch,description="loss at every epoch")
        # 
        #         
        # writer.add_scalars("Accuracy",{"Train/"+str(cfg.fold):train_acc*100},epoch,description="Accuracy at every epoch")
        # writer.add_scalars("Accuracy",{"Test/"+str(cfg.fold):test_acc*100},epoch,description="Accuracy at every epoch")
        # 
        # #writer.add_scalar('Loss/test', test_loss, epoch)
        # #writer.add_scalar('Loss/train', train_loss, epoch)
        # #writer.add_scalar('Accuracy/test', test_acc*100, epoch)
        # #writer.add_scalar('Accuracy/train', train_acc*100, epoch)
        # #writer.add_scalar('AccNum/train', train_correct, epoch) 
        # #writer.add_scalar('AccNum/test', correct, epoch) 
        # 
        # writer.add_scalars("AccuracySamples",{"Train/"+str(cfg.fold):train_correct},epoch)
        # writer.add_scalars("AccuracySamples",{"Test/"+str(cfg.fold):correct},epoch)
        # # writer.add_scalars("Train Loss/Accuracy",{'Loss':train_loss},epoch)
        # # writer.add_scalars("Train Loss/Accuracy",{'Accuracy':train_acc*100},epoch)
        # # 
        # # writer.add_scalars("Test Loss/Accuracy",{'Loss':test_loss},epoch)
        # # writer.add_scalars("Test Loss/Accuracy",{'Accuracy':test_acc*100},epoch)
        #     
        # 
        # writer.add_scalars("Loss/"+str(cfg.fold),{"Train":train_loss},epoch,description="loss at every epoch")
        # writer.add_scalars("Loss/"+str(cfg.fold),{"Test":test_loss},epoch,description="loss at every epoch")
        # writer.add_scalars("Accuracy/"+str(cfg.fold),{"Train":train_acc*100},epoch,description="Accuracy at every epoch")
        # writer.add_scalars("Accuracy/"+str(cfg.fold),{"Test":test_acc*100},epoch,description="Accuracy at every epoch")
        # writer.add_scalars("AccuracySamples/"+str(cfg.fold),{"Train":train_correct},epoch,description="TP/ALL at every epoch")
        # writer.add_scalars("AccuracySamples/"+str(cfg.fold),{"Test":correct},epoch,description="TP/ALL at every epoch")
        #Call flush() method to make sure that all pending events have been written to disk.
        writer.flush()

In [5]:
class EarlyStopping:
    def __init__(self, patience=7, verbose=False, delta=0, path='checkpoint.pth'):
        """
        Args:
            patience (int): How many epochs to wait after last time the monitored metric improved.
                            Default: 7
            verbose (bool): If True, prints a message for each validation loss improvement. 
                            Default: False
            delta (float): Minimum change in the monitored metric to qualify as an improvement.
                           Default: 0
            path (str): Path for the checkpoint to be saved to.
                        Default: 'checkpoint.pth'
        """
        self.patience = patience
        self.verbose = verbose
        self.counter = 0
        self.best_score = None
        self.early_stop = False
        self.val_loss_min = float('inf')
        self.delta = delta
        self.path = path

    def __call__(self, EPOCH, OPTM, STARTED_RUN, FOLD, TEST_ACC, val_loss, model):
        score = -val_loss
        text = ""
        if self.best_score is None:
            self.best_score = score
            text = self.save_checkpoint( EPOCH, OPTM, STARTED_RUN, FOLD, TEST_ACC, val_loss, model)
        elif score < self.best_score + self.delta:
            self.counter += 1
            if self.verbose:
                text = f'EarlyStopping counter: {self.counter} out of {self.patience}'
                print(text)
                
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_score = score
            text = self.save_checkpoint( EPOCH, OPTM, STARTED_RUN, FOLD, TEST_ACC, val_loss, model)
            self.counter = 0
        return text

    def save_checkpoint(self, EPOCH, OPTM, STARTED_RUN, FOLD, TEST_ACC, val_loss, model):
        """Saves model when validation loss decreases."""
        if self.verbose:
            text = f'Test Loss decreased ({self.val_loss_min:.6f} --> {val_loss:.6f}).'
            print(text, end="")
        #torch.save(model.state_dict(), self.path)
            torch.save({
                    'epoch': EPOCH,
                    'model_state_dict': model.state_dict(),
                    'optimizer_state_dict': OPTM.state_dict(),
                    'loss': val_loss,
                    'accuracy': TEST_ACC*100,
                }, self.path)#f"BestModel/{STARTED_RUN}/{FOLD}_ES_best_model.pth")
            
            
            self.val_loss_min = val_loss
            return text



In [25]:
#fold_results = np.array(cfg1.fold_test_acc)
def metrics_calc(cfg)->None:
        
    fold_results_final =  cfg.fold_test_acc_best
    print(fold_results_final)
    #print(fold_results_final.shape)
    # Calculate median, average, and plot boxplot
    median_accuracy = np.median(fold_results_final)
    average_accuracy = np.mean(fold_results_final)
    
    print(f"Median Accuracy: {median_accuracy:.3f}")
    print(f"Average Accuracy: {average_accuracy:.3f}")
    
    # Plot boxplot of accuracies
    fig, axs = plt.subplots(1,2,figsize=(7,3))
    fig.suptitle(f"Median Accuracy: {median_accuracy:.3f}" +f"  Average Accuracy: {average_accuracy:.3f}" )
    axs[0].boxplot(fold_results_final); axs[0].set_title('Cross-Validation Accuracy'); axs[0].set_ylabel('Accuracy');
    
    axs[1].stem(fold_results_final);axs[1].grid("on")
    fig.tight_layout()
    #plt.show()